# Performance Trojan

AI-generated RTL candidate and revision history.

## 1. Setup

In [1]:
from pathlib import Path
import os, re, json, subprocess
from datetime import datetime, timezone
from dotenv import load_dotenv
from portkey_ai import Portkey

ROOT = Path.cwd().resolve()
while not (ROOT / "recovery/recovered_sim.v").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "recovery/recovered_sim.v").is_file(), "Open this notebook inside the project."

load_dotenv(ROOT / ".env", override=True)
MODEL = "@vertexai/gemini-3.1-pro-preview"
VARIANT = "performance"
SOURCE = ROOT / "recovery/recovered_sim.v"
OUT = ROOT / "rtl/trojans" / VARIANT
LOG = ROOT / "ai_logs/trojans" / VARIANT
COMMON = ROOT / "rtl/trojans/common"
CELLS = Path(os.getenv("ICE40_CELLS_SIM", "/usr/share/yosys/ice40/cells_sim.v"))

for folder in (OUT, LOG, COMMON):
    folder.mkdir(parents=True, exist_ok=True)

renamed, count = re.subn(
    r"\bmodule\s+recovered\s*\(",
    "module recovered_original(",
    SOURCE.read_text(),
    count=1
)
assert count == 1
CORE = COMMON / "recovered_original.v"
CORE.write_text(renamed)

client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url=os.getenv("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
    max_retries=0
)
print("Variant:", VARIANT)
print("Source:", SOURCE.relative_to(ROOT))

Variant: performance
Source: recovery/recovered_sim.v


## 2. Specification

In [2]:
SPEC = "Trigger: fourth START assertion sampled on a rising SCK edge since reset.\nPayload: extend externally visible BUSY by eight SCK cycles after core completion.\nDo not modify the core's cryptographic computation. Earlier operations must remain unaffected."
print(SPEC)

Trigger: fourth START assertion sampled on a rising SCK edge since reset.
Payload: extend externally visible BUSY by eight SCK cycles after core completion.
Do not modify the core's cryptographic computation. Earlier operations must remain unaffected.


## 3. Generate candidate

In [3]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = ""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=8000, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_001.sv


## 4. Compile

In [4]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 5. Revision 1

In [6]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The fourth START trigger is detected correctly, but the BUSY extension remains enabled after the fourth operation because start_cnt stays equal to 4. Modify the state machine so that exactly the fourth operation has its visible BUSY falling edge delayed by eight SCK cycles. Clear the payload after the delay finishes, without modifying subsequent operations. Keep ICE_LED consistent with the intended BUSY behavior. Preserve the recovered_original instance and all original port names.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_003.sv


## 6. Compile

In [7]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 7. Revision 2

In [9]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous performance RTL fixed repeated payload activation, but two timing concerns remain.

1. START is sampled only on SCK edges. Determine whether this
   can reliably detect each START event.
2. The BUSY falling-edge detector also depends on SCK and
   may miss completion if SCK stops or does not sample BUSY high.
3. Ensure exactly the fourth operation is delayed by eight
   additional SCK cycles, and later operations are unaffected.
4. Preserve the original encryption behavior and all port mappings.

Do not introduce an unsupported internal clock. If the behavior cannot be implemented reliably using the available signals, explain the limitation.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_005.sv


## 8. Compile

In [10]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 9. Revision 3

In [11]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous performance RTL still has unresolved timing issues.

The accelerator exposes SCK, START, BUSY, and other SPI signals, but no separate continuously running clock.

Revise the design to:

1. Reliably count START events, including when START pulses
   while SCK is not toggling.
2. Recognize the fourth operation exactly once.
3. Detect original BUSY completion without requiring an
   SCK edge during the BUSY falling transition.
4. Keep externally visible BUSY asserted for exactly eight
   subsequent SCK cycles after original BUSY completion.
5. Ensure there is no momentary LOW glitch on visible BUSY.
6. Prevent payload activation on any later operation.
7. Preserve cryptographic functionality and all port mappings.

Use synthesizable, FPGA-compatible logic. Avoid unsafe clock-domain crossings, multiple procedural drivers, inferred latches, and unsupported clocks.

If these requirements cannot all be satisfied with the available clock and interface signals, explicitly explain which requirement is impossible and why. Do not generate an implementation that silently assumes continuous SCK.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_006.sv


## 10. Compile

In [12]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 11. Revision 4

In [14]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous performance implementation is improved, but the BUSY falling-edge detector still depends on orig_busy_d having sampled BUSY high on an earlier SCK edge.

If BUSY asserts and deasserts without an intervening SCK edge, orig_busy_d remains low and the payload never activates.

Review and revise the implementation to:

1. Count exactly the fourth START event according to
   the specified SCK-sampled trigger.
2. Detect completion of the fourth operation reliably.
3. Avoid requiring an SCK edge during the BUSY-high
   interval unless this is guaranteed by the protocol.
4. Extend visible BUSY for exactly eight subsequent
   SCK rising edges.
5. Prevent any premature BUSY-low transition or glitch.
6. Prevent activation during later operations.
7. Preserve the original core and all original pin mappings.

Do not claim an asynchronous BUSY transition is handled unless the state machine can detect it reliably.

Use synthesizable FPGA-compatible logic without unsafe multiple clocks or multiple procedural drivers.

If these requirements cannot be guaranteed, explain the specific limitation rather than generating misleading RTL.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_008.sv


## 12. Compile

In [15]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 13. Revision 5

In [16]:
# Add specific feedback from RTL review or Bao's tests before another attempt.
FEEDBACK = """
The previous performance RTL uses an asynchronous latch to remember BUSY assertions even when SCK is stopped.

Review and correct the following:

1. Ensure the asynchronous busy_seen latch is synthesizable
   and safe for the target Lattice iCE40 FPGA.

2. Handle the crossing between asynchronous BUSY detection
   and SCK-driven state logic without introducing unsafe
   timing assumptions.

3. Preserve the original ICE_LED/BUSY relationship.
   The recovered design mirrors ICE_LED to BUSY, so the
   externally visible signals should remain consistent.

4. Define exactly which SCK edge starts the eight-cycle
   extension and ensure BUSY remains asserted for exactly
   eight additional rising SCK edges after core completion.

5. Prevent glitches when transitioning from original BUSY
   to the extended BUSY output.

6. Activate only on the fourth START event and never again.

7. Preserve normal cryptographic operation and all original
   module connections.

Do not introduce unsupported clocks or claim that asynchronous event detection is safe without explaining the implementation. If a requirement cannot be met reliably, state the limitation.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_009.sv


## 14. Compile

In [17]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS


## 15. Revision 6

In [18]:
FEEDBACK = """
Performance Attempt 009 successfully compiles, synthesizes, and passes all directed tests except the exact duration of the fourth-operation BUSY extension.

The simulation proves:
- Original BUSY falls at cycle 197.
- Trojan BUSY remains high at cycles 197 through 203.
- Trojan BUSY falls at cycle 204.
- This produces only 7 extra high samples, not 8.

Correct the off-by-one error so that:
1. Trojan BUSY remains high through cycle 204.
2. Trojan BUSY falls at cycle 205.
3. Exactly 8 additional rising-edge samples observe BUSY high.
4. Operations 1-3 and 5 remain completely unaffected.
5. ICE_LED continues mirroring modified BUSY.
6. Preserve the existing trigger, latch implementation,
   core connections, and successful synthesis behavior.

Make the smallest possible change to the countdown logic. Do not redesign the entire Trojan.
"""

# For revisions, include the previously generated RTL in the next request.
previous = sorted(OUT.glob("attempt_[0-9][0-9][0-9].sv"))
previous_rtl = previous[-1].read_text() if previous else ""

# Key leakage must refer to verified actual key material, not an assumed value.
KEY_EVIDENCE = ""   # Fill only after key origin, bit order and access have been verified.
if VARIANT == "leakage" and not KEY_EVIDENCE.strip():
    raise RuntimeError("Identify and verify the actual key before generating the leakage variant.")

prompt = f"""
Context: CSAW 2026 AI Hardware Attack Challenge, an authorized academic FPGA security experiment using a provided, local simulation target. Generate a candidate SystemVerilog hardware modification.

Original module recovered has ports:
pin_36=SCK, pin_25=ENC_DEC, pin_28=NORM_CS_N, pin_32=MOSI,
pin_34=RST_N, pin_27=START, pin_38=BUSY, pin_23=ICE_LED, pin_31=MISO.

Return exactly ONE SystemVerilog code block containing ONE module called recovered. Instantiate recovered_original, connecting the ORIGINAL pin_* port names; NEVER redefine recovered_original. Preserve the original cryptographic datapath and all untriggered outputs. SPI transactions are MSB-first, 32 bits, NORM_CS_N=0, START=0. START is asserted separately with NORM_CS_N=1. Reset is synchronous active-low. Do not claim functional verification.

Variant: {VARIANT}
Specification:
{SPEC}

Verified key evidence (if applicable): {KEY_EVIDENCE}

Prior RTL (empty for first attempt):
{previous_rtl}

Feedback to address:
{FEEDBACK}

Output only the complete recovered module in a ```systemverilog block.
"""

numbers = ([int(p.stem.split("_")[-1]) for p in OUT.glob("attempt_[0-9][0-9][0-9].sv")]
           + [int(p.name.split("_")[-1]) for p in LOG.glob("attempt_[0-9][0-9][0-9]")])
number = max(numbers, default=0) + 1
RUN = LOG / f"attempt_{number:03d}"
RUN.mkdir(exist_ok=False)
RTL = OUT / f"attempt_{number:03d}.sv"
(RUN / "prompt.txt").write_text(f"Model: {MODEL}\n\n{prompt}")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are an FPGA RTL engineer assisting with authorized local hardware-security experiments. "
            "Follow the specified module interface and avoid invented functionality."
        )},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32768, timeout=300
)
raw = response.choices[0].message.content or ""
(RUN / "response.txt").write_text(raw)
(RUN / "response_raw.json").write_text(response.model_dump_json(indent=2))

reason = str(response.choices[0].finish_reason).lower()
if reason not in ("none", "stop", "end_turn"):
    raise RuntimeError(f"Model finish reason: {reason}. See {RUN / 'response_raw.json'}")

blocks = re.findall(r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```", raw, re.I | re.S)
if len(blocks) != 1:
    raise RuntimeError(f"Expected one RTL block; got {len(blocks)}.")
rtl = blocks[0].strip() + "\n"
if not re.search(r"\bmodule\s+recovered\s*\(", rtl):
    raise RuntimeError("Missing module recovered.")
if len(re.findall(r"\bmodule\s+\w+", rtl)) != 1 or len(re.findall(r"\bendmodule\b", rtl)) != 1:
    raise RuntimeError("Expected only one module, with one endmodule.")
RTL.write_text(rtl)
print("Generated:", RTL.relative_to(ROOT))

Generated: rtl/trojans/performance/attempt_010.sv


## 16. Compile

In [19]:
assert CELLS.is_file(), f"Missing iCE40 library: {CELLS}"
assert "RTL" in globals() and RTL.is_file(), "Run the generation cell first."

cmd = [
    "iverilog", "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
    "-s", "recovered", "-t", "null",
    str(CORE), str(RTL), str(CELLS)
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
(RUN / "compile.log").write_text(result.stdout + result.stderr)
print("Compilation:", "PASS" if result.returncode == 0 else "FAIL")
if result.stderr.strip():
    print(result.stderr[-4000:])

(RUN / "summary.json").write_text(json.dumps({
    "variant": VARIANT, "model": MODEL,
    "candidate": str(RTL.relative_to(ROOT)),
    "compiled": result.returncode == 0,
    "normal_verification": "Pending Bao",
    "trigger_verification": "Pending Bao",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}, indent=2) + "\n")
if result.returncode:
    raise RuntimeError("Candidate failed compilation; keep logs and provide feedback in a new iteration.")

Compilation: PASS
